# GRPO on Qwen2.5-1.5B · GSM8K · one Colab A100

Runtime → Change runtime type → **A100 GPU**. Then Runtime → Run all.

Pipeline: base eval → GRPO training → trained eval → report (fills the headline + README) → push results to GitHub.

Optional Colab secrets (🔑 sidebar): `GITHUB_TOKEN` (fine-grained, contents read/write on this repo) to push results; `HF_TOKEN` to push the model; `WANDB_API_KEY` for live tracking.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
REPO = "UbajakaCJ/grpo-gsm8k"   # <- change if you renamed the repo
%cd /content
!git clone https://github.com/{REPO}.git
%cd /content/grpo-gsm8k

In [ ]:
# vLLM first so it settles the torch version, then the package.
!pip install -q vllm
!pip install -q -e ".[train,dev]"
!python -c "import torch, trl, vllm; print(torch.__version__, trl.__version__, vllm.__version__)"

In [ ]:
import os
from google.colab import userdata
for key in ("HF_TOKEN", "WANDB_API_KEY", "GITHUB_TOKEN"):
    try:
        os.environ[key] = userdata.get(key)
        print(f"{key}: set")
    except Exception:
        print(f"{key}: not set (optional)")

### 1 · Smoke test (≈5 min)
Tiny model, 4 steps, 32 eval problems — catches environment problems before the long run.

In [ ]:
!make test && make smoke

### 2 · Full run
Edit `configs/colab_a100.yaml` or pass overrides, e.g. `!python -m grpo_gsm8k.train --config configs/colab_a100.yaml train.max_steps=300`.
Add `train.report_to=wandb` to log to Weights & Biases. If you hit OOM, set `model.lora.enabled=true` or lower `train.vllm_gpu_memory_utilization`.

In [ ]:
!make all CONFIG=configs/colab_a100.yaml 2>&1 | tee results/run.log

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open("results/headline.md").read()))
import os
if os.path.exists("results/training_curves.png"):
    display(Image("results/training_curves.png"))

### 3 · Push results to GitHub
Commits `results/` and the regenerated README. Checkpoints stay out of git (see `.gitignore`); set `hub.push_to_hub: true` + `hub.repo_id` in the config to push the model to the HF Hub.

In [ ]:
import os, subprocess
token = os.environ.get("GITHUB_TOKEN")
assert token, "Add a GITHUB_TOKEN Colab secret first"
!git config user.name "Chijioke Ubajaka"
!git config user.email "ubajaka.chijioke@users.noreply.github.com"
!git add results README.md
!git commit -m "Add GRPO GSM8K results" || echo "nothing to commit"
subprocess.run(["git", "push", f"https://{token}@github.com/{REPO}.git", "HEAD:main"], check=True)